In [ ]:
# import datasets
# from datasets import load_dataset
from utils import DATA_DIR

#ds = load_dataset("BramVanroy/wikipedia_culturax_dutch", "100k", split="train")
with open(DATA_DIR/"clean_corpus.txt", "r", encoding="utf-8") as f:
    custom_corpus = f.read().splitlines()
    custom_corpus = custom_corpus[:len(custom_corpus)//4]  # use part for testing
print(f"Length custom corpus: {len(custom_corpus)}")

In [5]:
# what i need:
# (freq, verb, relation, dep1)

In [2]:
import spacy
nlp = spacy.load("nl_core_news_md", disable=["ner", "textcat"])
test_sent = ("Dit is een testzin. De jongen gooit de bal naar de hond. "
             "Hij gooide hem op staande voet terug. Ik geef Sinterklaas een trap.")
doc = nlp(test_sent)
for sent in doc.sents:
    if sent.root.pos_ == "VERB":
        print("root of sent:", sent.root.lemma_, sent.root.pos_)
        print([(child.lemma_, child.pos_, child.dep_) for child in sent.root.children])
        
    # for token in sent:
    #     print(f"{token.text}: {token.pos_}, {token.tag_}, {token.head.text}:{token.dep_}")
    #     # we check dependencies
    #     print(f"\tHead: {token.head.text}, Dep: {token.dep_}")
    

root of sent: gooit VERB
[('jongen', 'NOUN', 'nsubj'), ('bal', 'NOUN', 'obj'), ('hond', 'NOUN', 'obl'), ('.', 'PUNCT', 'punct')]
root of sent: gooien VERB
[('hij', 'PRON', 'nsubj'), ('hem', 'PRON', 'obj'), ('voet', 'NOUN', 'obl'), ('terug', 'ADV', 'compound:prt'), ('.', 'PUNCT', 'punct')]
root of sent: geven VERB
[('ik', 'PRON', 'nsubj'), ('Sinterklaas', 'PROPN', 'iobj'), ('trap', 'NOUN', 'obj'), ('.', 'PUNCT', 'punct')]


In [15]:
custom_corpus_equal = 0
for sent in custom_corpus:
    if len(sent.split("\n")) > 1:
        custom_corpus_equal += 1
custom_corpus_equal

0

In [12]:
# we import counter
from tqdm import tqdm
from collections import Counter, defaultdict
verb_relation_counter = Counter()
for doc in nlp.pipe(tqdm(custom_corpus), batch_size=1000):
    for sent in doc.sents:
        if sent.root.pos_ == "VERB":
            verb = sent.root.lemma_
            for child in sent.root.children:
                relation = child.dep_
                if relation in {"nsubj", "obj", "iobj", "obl"}:
                    verb_relation_counter[(verb, relation+"#"+child.lemma_)] += 1

100%|██████████| 100727/100727 [23:05<00:00, 72.71it/s] 


In [13]:
verb_relation_counter.most_common(20)

[(('publiceren', 'obl#voor'), 20653),
 (('tellen', 'obj#inwoner'), 4344),
 (('spelen', 'nsubj#hij'), 4093),
 (('maken', 'nsubj#hij'), 3650),
 (('krijgen', 'nsubj#hij'), 2368),
 (('winnen', 'nsubj#hij'), 2322),
 (('tellen', 'nsubj#plaats'), 2317),
 (('komen', 'nsubj#soort'), 2282),
 (('werken', 'nsubj#hij'), 2256),
 (('komen', 'nsubj#hij'), 2147),
 (('maken', 'obj#debuut'), 2036),
 (('gaan', 'nsubj#hij'), 1863),
 (('nemen', 'nsubj#hij'), 1641),
 (('rekenen', 'obl#familie'), 1569),
 (('schrijven', 'nsubj#hij'), 1500),
 (('worden', 'nsubj#hij'), 1419),
 (('trouwen', 'nsubj#hij'), 1410),
 (('krijgen', 'nsubj#ze'), 1391),
 (('beginnen', 'nsubj#hij'), 1351),
 (('overlijden', 'obl#leeftijd'), 1344)]

In [14]:
with open(DATA_DIR/"verb_relation_counts.txt", "a", encoding="utf-8") as f: # we add instead of overwriting
    for (verb, relation_dep), freq in verb_relation_counter.items():
        f.write(f"{freq}#{verb}#{relation_dep}\n")

In [25]:
from tqdm import tqdm
tagged_texts = []
for doc in nlp.pipe(tqdm(custom_corpus), batch_size=1000):
    for sent in doc.sents:
        tagged_texts.append(sent)

100%|██████████| 402908/402908 [1:35:56<00:00, 69.99it/s]  


In [26]:
# we save as a tagged corpus, and as a pickle file
import pickle
with open(DATA_DIR/"tagged_corpus.txt", "w", encoding="utf-8") as f:
    for sent in tagged_texts:
        for token in sent:
            f.write(f"{token.text}\t{token.pos_}\t{token.tag_}\t{token.dep_}\n")
        f.write("\n")

with open(DATA_DIR/"tagged_corpus.pkl", "wb") as f:
    pickle.dump(tagged_texts, f)

NotImplementedError: [E112] Pickling a span is not supported, because spans are only views of the parent Doc and can't exist on their own. A pickled span would always have to include its Doc and Vocab, which has practically no advantage over pickling the parent Doc directly. So instead of pickling the span, pickle the Doc it belongs to or use Span.as_doc to convert the span to a standalone Doc object.

In [28]:
with open(DATA_DIR/"tagged_corpus.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
for line in lines[0:20]:
    print(line.strip())

Charax	NOUN	N|soort|ev|basis|onz|stan	amod
metae	NOUN	N|soort|ev|basis|onz|stan	nsubj
is	AUX	WW|pv|tgw|ev	cop
een	DET	LID|onbep|stan|agr	det
straalvinnige	ADJ	ADJ|prenom|basis|met-e|stan	nsubj
vissensoort	NOUN	N|soort|ev|basis|onz|stan	ROOT
uit	ADP	VZ|init	case
de	DET	LID|bep|stan|rest	det
familie	NOUN	N|soort|ev|basis|zijd|stan	obl
van	ADP	VZ|init	case
de	DET	LID|bep|stan|rest	det
karperzalmen	NOUN	N|soort|mv|basis	nmod
(	PUNCT	LET	punct
Characidae	PROPN	N|eigen|ev|basis|onz|stan	nmod
)	PUNCT	LET	punct
.	PUNCT	LET	punct

De	DET	LID|bep|stan|rest	det
wetenschappelijke	ADJ	ADJ|prenom|basis|met-e|stan	amod
naam	NOUN	N|soort|ev|basis|zijd|stan	nsubj:pass


In [32]:
clean_tagged_texts = [sent for sent in tagged_texts if len(sent) > 3]
with open(DATA_DIR/"clean_tagged_corpus.txt", "w", encoding="utf-8") as f:
    for sent in clean_tagged_texts:
        for token in sent:
            f.write(f"{token.text}\t{token.pos_}\t{token.tag_}\t{token.dep_}\n")
        f.write("\n")

In [39]:
with open(DATA_DIR/"clean_tagged_corpus.txt", "w", encoding="utf-8") as f:
    for sent in clean_tagged_texts:
        for token in sent:
            f.write(f"{token.text}\t{token.pos_}\t{token.dep_}\n")
        f.write("\n")

In [ ]:
serializable_sents = []
for sent in tqdm(clean_tagged_texts):
    serializable_sent = []
    for token in sent:
        serializable_sent.append((token.text, token.pos_, token.tag_, token.dep_))
    serializable_sents.append(serializable_sent)

 24%|██▍       | 1321618/5522809 [01:59<8:01:08, 145.53it/s]

In [ ]:
with open(DATA_DIR/"tagged_corpus.pkl", "wb") as f:
    pickle.dump(serializable_sents, f)

# Vector extraction
we will create verb-anchored vectors from pos
<VERB nsubj obj iobj obl >

In [28]:
with open(DATA_DIR/"clean_corpus.txt", "r", encoding="utf-8") as f:
    custom_corpus = f.read().splitlines()
    custom_corpus = custom_corpus[:len(custom_corpus)//10000]  # use part for testing

In [23]:
import spacy
nlp = spacy.load("nl_core_news_sm", disable=["ner", "textcat"])
test_sent = ("Dit is een testzin. De jongen gooit de bal naar de hond. "
             "Hij gooide hem op staande voet terug. Ik geef Sinterklaas een trap.")
doc = nlp(test_sent)
targets = ["nsubj", "obj", "obj2", "obl"]
for sent in doc.sents:
    if sent.root.pos_ == "VERB":
        children_deps = {target:None for target in targets}
        for child in sent.root.children:
            if child.dep_ in targets:
                if child.dep_ == "obj" and children_deps["obj"]:
                    children_deps["obj2"] = child.lemma_
                else:
                    children_deps[child.dep_] = child.lemma_
        vector = (sent.root.lemma_,children_deps["nsubj"], children_deps["obj"], children_deps["obj2"], children_deps["obl"])

root of De jongen gooit de bal naar de hond.: gooit VERB
('gooit', 'jongen', 'bal', None, 'hond')
root of Hij gooide hem op staande voet terug.: gooien VERB
('gooien', 'hij', 'hem', None, 'voet')
root of Ik geef Sinterklaas een trap.: geven VERB
('geven', 'ik', 'Sinterklaas', 'trap', None)


In [25]:
S = nlp.vocab.strings
DEP_NSUBJ = S["nsubj"]
DEP_OBJ   = S["obj"]
DEP_OBL   = S["obl"]
print(DEP_OBJ, DEP_NSUBJ)

434 429


In [26]:
from itertools import islice
import spacy

nlp = spacy.load("nl_core_news_sm", disable=["ner", "textcat"])


# --- map labels to ints once ---
S = nlp.vocab.strings
DEP_NSUBJ = S["nsubj"]
DEP_OBJ   = S["obj"]
DEP_OBL   = S["obl"]
ROOT_VERB = S["VERB"]

def extract_vectors(doc):
    results = []
    for sent in doc.sents:
        root = sent.root
        if root.pos != ROOT_VERB:  # or: if root.pos_ != "VERB"
            continue

        nsubj = obj = obj2 = obl = None
        filled = 0  # how many slots filled

        for child in root.children:
            d = child.dep
            if d == DEP_NSUBJ and nsubj is None:
                nsubj = child.lemma_
                filled += 1
            elif d == DEP_OBJ:
                if obj is None:
                    obj = child.lemma_
                    filled += 1
                elif obj2 is None: # no reliable way to discriminate without extra processing, we simplify to this
                    obj2 = child.lemma_
                    filled += 1
            elif d == DEP_OBL and obl is None:
                obl = child.lemma_
                filled += 1

            if filled == 4:  # all targets filled
                break

        results.append((root.lemma_, nsubj, obj, obj2, obl))
    return results

# --- fast, batched processing ---
texts = [
    "Dit is een testzin. De jongen gooit de bal naar de hond. "
    "Hij gooide hem op staande voet terug. Ik geef Sinterklaas een trap."
]

# Use n_process>1 if you have multiple cores; tune batch_size for throughput
all_vectors = []
for doc in nlp.pipe(texts, batch_size=1000, n_process=2):
    all_vectors.extend(extract_vectors(doc))


In [42]:
all_vectors = []
with open(DATA_DIR/"clean_corpus.txt", "r", encoding="utf-8") as f:
    custom_corpus = f.read().splitlines()
    # custom_corpus = custom_corpus[:len(custom_corpus)//50]  # use part for testing
for doc in nlp.pipe(tqdm(custom_corpus), batch_size=1000, n_process=-1):
    all_vectors.extend(extract_vectors(doc))



  0%|          | 0/402908 [00:00<?, ?it/s]

  2%|▏         | 10000/402908 [00:00<00:03, 98327.89it/s]

  8%|▊         | 32000/402908 [02:39<34:04, 181.43it/s]  

  8%|▊         | 32141/402908 [02:39<33:51, 182.52it/s]

  9%|▉         | 36270/402908 [02:40<26:01, 234.83it/s]

 10%|█         | 41244/402908 [02:40<18:34, 324.65it/s]

 11%|█▏        | 46023/402908 [02:40<13:17, 447.44it/s]

 12%|█▏        | 50292/402908 [03:29<27:20, 214.98it/s]

 14%|█▎        | 54651/402908 [03:29<19:35, 296.16it/s]

 15%|█▍        | 59120/402908 [03:29<13:49, 414.55it/s]

 16%|█▌        | 63461/402908 [03:29<09:47, 577.42it/s]

 17%|█▋        | 67624/402908 [04:18<25:30, 219.07it/s]

 18%|█▊        | 72090/402908 [04:19<17:37, 312.69it/s]

 19%|█▉        | 77310/402908 [04:19<11:35, 468.40it/s]

 20%|██        | 81794/402908 [05:10<25:58, 206.09it/s]

 22%|██▏       | 87136/402908 [05:10<17:06, 307.57it/s]

 23%|██▎       | 92653/402908 [05:10<11:19, 456.62it/s]

 24%|██▍       | 97618/402908 [06:02<2

In [3]:
import pickle
with open('data/all_vectors_cleaned.pkl', 'wb') as p:
    pickle.dump(all_vectors, p)

NameError: name 'all_vectors' is not defined

EOFError: Ran out of input

In [46]:
# from this we can build counters as well
counted_verbs = Counter()
for (verb, nsubj, obj, obj2, obl) in tqdm(all_vectors):
    for role, value in zip(["nsubj", "obj", "obj2", "obl"], [nsubj, obj, obj2, obl]):
        if value:
            counted_verbs[(verb, f"{role}#{value}")] += 1
    



  0%|          | 0/3046596 [00:00<?, ?it/s]

  2%|▏         | 51851/3046596 [00:00<00:05, 516891.17it/s]

  3%|▎         | 104425/3046596 [00:00<00:05, 521074.73it/s]

  5%|▌         | 157927/3046596 [00:00<00:05, 527031.69it/s]

  7%|▋         | 210631/3046596 [00:00<00:05, 520763.22it/s]

  9%|▊         | 262718/3046596 [00:00<00:05, 516260.85it/s]

 10%|█         | 316170/3046596 [00:00<00:05, 521342.31it/s]

 12%|█▏        | 368315/3046596 [00:00<00:05, 517329.78it/s]

 14%|█▍        | 420058/3046596 [00:00<00:05, 504157.59it/s]

 15%|█▌        | 470536/3046596 [00:00<00:05, 482741.53it/s]

 17%|█▋        | 519242/3046596 [00:01<00:05, 483070.36it/s]

 19%|█▊        | 569571/3046596 [00:01<00:05, 488030.27it/s]

 20%|██        | 618478/3046596 [00:01<00:05, 485342.19it/s]

 22%|██▏       | 667082/3046596 [00:01<00:05, 469198.52it/s]

 23%|██▎       | 714136/3046596 [00:01<00:04, 469111.48it/s]

 25%|██▌       | 762333/3046596 [00:01<00:04, 472043.05it/s]

 27%|██▋       | 811760/

In [47]:
with open(DATA_DIR/"verb_relation_counts_all.txt", "w", encoding="utf-8") as f: # we add instead of overwriting
    for (verb, relation_dep), freq in counted_verbs.items():
        f.write(f"{freq}#{verb}#{relation_dep}\n")

In [ ]:
# we put all the vectors in a tensor space with modes "nsubj", "obj", "obj2", "obl"


# Open Subtitles

In [10]:
from datasets import load_dataset

ds = load_dataset("sentence-transformers/parallel-sentences-opensubtitles", "en-nl", split="train", streaming=True)
dataset_path="epfml/FineWeb2-HQ"
dataset_config="nld_Latn"
og_ds = load_dataset(dataset_path, dataset_config, split="train", streaming=True)


Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1205 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/163 [00:00<?, ?it/s]

In [15]:
1_000_000 == 1000000

True

In [11]:
def gen_texts(dataset):
    for ex in dataset:
        # each ex is a dict like {"text": "...", ...}
        txt = ex.get("text")
        if txt:
            yield txt

In [12]:
for i, text in enumerate(gen_texts(og_ds)):
    print(text)
    if i >= 5:
        break

De Europese vlag wordt gevormd door 12 gouden sterren in een cirkel tegen een blauwe achtergrond. De sterren symboliseren de idealen van eenheid, solidariteit en harmonie tussen de volkeren van Europa.
De sterren in een cirkel vormen een teken van eenheid en hebben niets te maken met het aantal EU-landen.
Geschiedenis van de Europese vlag
In de jaren daarna moedigde de Raad van Europa de nieuwe Europese instellingen ertoe aan de vlag over te nemen.
In 1983 nam het Europees Parlement de vlag over. In 1985 voerden de EU-regeringsleiders de vlag in als officieel symbool van de Europese Unie (die toen nog Europese Gemeenschap heette). Alle Europese instellingen gebruiken nu een eigen logo.


Bodog Poker
Bodog omhooggeschoten op de texas holdem scène in '04, waardoor haar mix van spel speelt informatie (hun eigen sportieve activiteiten e-boek en ook casino zijn er al doet zijn ding voor een tijd) en de rijke levensstijl van exploitant Calvin Ayre. Zij hebben gestuurd spelers gevarieerde van

In [ ]:
for i, text in enumerate(gen_texts(ds)):
    print(text)
    if i >= 5:
        break

In [9]:
# we adapt the code to work on a directory of text files
import os
def gen_texts_from_dir(directory):
    for filename in os.listdir(directory):
        if filename.endswith(".txt.sent"):
            filepath = os.path.join(directory, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                # there are many short lines, we join into sentences/paragraphs ending with .
                text = f.read().replace("\n", " ")
                for sent in text.split("."):
                    sent = sent.strip()
                    if sent:
                        yield sent + "."



In [10]:
# we test this with the data/karrewiet directory
for i, text in enumerate(gen_texts_from_dir(DATA_DIR/"karrewiet")):
    print(i, text)
    if i >= 15:
        break

0 De spanning stijgt.
1 Morgen is de finale van het Junior Eurovisiesongfestival.
2 Voorspellen wie wint is niet makkelijk.
3 Ook niet voor Femke.
4 Ik vind het toch moeilijk om te voorspellen wie er zal winnen.
5 Dat is zo.
6 Dat is zo.
7 Ik vind iedereen goed.
8 Ik weet niet of België kan winnen.
9 Dat hangt natuurlijk van de mensen af.
10 Ik ga gewoon alles geven en er keihard van genieten.
11 En we zien wel wat het geeft.
12 Net als België moet ook Georgië morgen lang wachten.
13 Zij komen als voorlaatste op het podium.
14 * Candy music  *   * Candy music  *   Misschien ga ik wel een beetje nerveus worden.
15 Maar als je zo de intro van mijn liedje hebt.
